# Credit Scoring Orquestador

Notebook orquestador del homework. Toda la logica de negocio vive en `src/`.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [ ]:
import json

import pandas as pd
from sklearn.model_selection import train_test_split

from src.data import load_raw, validate_schema, create_features
from src.features import select_features_by_iv, build_woe_tables, transform_woe
from src.models import train_all_models, evaluate_models, save_model
from src.metrics import auc_roc, costo_total, build_scorecard

## 1. Carga y validacion

In [ ]:
data_path = PROJECT_ROOT / "data" / "raw" / "bankloan.csv"
assert data_path.exists(), "No existe data/raw/bankloan.csv en el repositorio"
assert data_path.suffix == ".csv", "El dataset de entrada debe estar en formato CSV"

df = load_raw(str(data_path))
assert not df.empty, "El dataset cargado esta vacio"
validate_schema(df)
df = create_features(df)
assert not df.empty, "El dataset quedo vacio despues del preprocesamiento"
df.head()

In [ ]:
feature_df = df.drop(columns=["customer"], errors="ignore")

X_train, X_test, y_train, y_test = train_test_split(
    feature_df.drop(columns=["Default"]),
    feature_df["Default"],
    test_size=0.2,
    random_state=42,
    stratify=feature_df["Default"],
)

print(X_train.shape, X_test.shape)

## 2. Seleccion de variables por IV

In [ ]:
train_with_target = X_train.assign(Default=y_train)
features_sel = select_features_by_iv(
    train_with_target,
    target="Default",
    threshold=0.1,
)
assert features_sel, "No se seleccionaron variables con el umbral de IV actual"
features_sel

## 3. Transformacion WoE

In [ ]:
woe_tables = build_woe_tables(
    train_with_target,
    features_sel,
    target="Default",
)

X_train_woe = transform_woe(X_train[features_sel], woe_tables)
X_test_woe = transform_woe(X_test[features_sel], woe_tables)

X_train_woe.head()

In [ ]:
scorecard_df = build_scorecard(woe_tables)
scorecard_df.head()

## 4. Entrenamiento y evaluacion

In [ ]:
models = train_all_models(X_train_woe, y_train)
tabla_auc = evaluate_models(models, X_test_woe, y_test)
tabla_auc

In [ ]:
mejor_nombre = tabla_auc.iloc[0]["Modelo"]
mejor_modelo = models[mejor_nombre]

metadata = {
    "model_name": mejor_nombre,
    "version": "1.0",
    "author": "Arenas",
    "dataset": "data/raw/bankloan.csv",
    "n_train_samples": int(len(y_train)),
    "n_test_samples": int(len(y_test)),
    "features_selected": list(X_train_woe.columns),
    "hyperparameters": mejor_modelo.get_params(),
    "metrics": {
        "auc_test": round(auc_roc(mejor_modelo, X_test_woe, y_test), 4),
        "costo_total_test": int(
            costo_total(
                y_test.to_numpy(),
                mejor_modelo.predict_proba(X_test_woe)[:, 1],
                umbral=0.5,
            )
        ),
    },
}

model_path = save_model(
    mejor_modelo,
    path=str(PROJECT_ROOT / "models" / "baseline_v1"),
    metadata=metadata,
)
assert model_path.exists(), "El archivo del modelo no fue generado"
model_path

## 5. Verificacion de metadata

In [ ]:
metadata_path = PROJECT_ROOT / "models" / "baseline_v1" / "metadata.json"

assert metadata_path.exists(), "metadata.json no existe, revisa save_model()"

with open(metadata_path, encoding="utf-8") as f:
    meta = json.load(f)

campos_requeridos = [
    "model_name",
    "version",
    "saved_at",
    "author",
    "dataset",
    "n_train_samples",
    "n_test_samples",
    "features_selected",
    "hyperparameters",
    "metrics",
]

for campo in campos_requeridos:
    assert campo in meta, f"Campo faltante en metadata.json: {campo}"

metricas_requeridas = ["auc_test", "costo_total_test"]
for metrica in metricas_requeridas:
    assert metrica in meta["metrics"], f"Metrica faltante en metadata.json: {metrica}"

assert meta["author"] == "Arenas", "El author de metadata.json no coincide"
assert meta["dataset"] == "data/raw/bankloan.csv", "La ruta del dataset en metadata.json no coincide"
assert meta["n_train_samples"] + meta["n_test_samples"] == len(df), "El total de muestras en metadata.json no coincide con el dataset procesado"
assert isinstance(meta["features_selected"], list) and meta["features_selected"], "features_selected debe ser una lista no vacia"
assert isinstance(meta["hyperparameters"], dict) and meta["hyperparameters"], "hyperparameters debe ser un diccionario no vacio"

pd.DataFrame([meta])